In [1]:
import os

# ---------------------------------------------------------
# 0. HARDWARE ISOLATION & MEMORY PATCHES
# ---------------------------------------------------------
# Force PyTorch to ONLY see GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Fix PyTorch memory fragmentation on A100
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
# Force cuDNN to behave predictably
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.enabled = False

from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperForConditionalGeneration, 
    WhisperProcessor, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import PeftModel
from datasets import load_dataset

# ---------------------------------------------------------
# 1. SETUP PATHS & LOAD BASE MODEL + YOUR LORA ADAPTER
# ---------------------------------------------------------
model_id = "openai/whisper-small"
checkpoint_dir = "./my_whisper_checkpoint/checkpoint-200" 

processor = WhisperProcessor.from_pretrained(model_id, language="Hindi", task="transcribe")

quant_config = BitsAndBytesConfig(load_in_8bit=True)

# Because we hid GPU 1, device_map="auto" will now safely load 100% of the model onto GPU 0
model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.enable_input_require_grads()
model.config.use_cache = False

model = PeftModel.from_pretrained(model, checkpoint_dir, is_trainable=True)

print("Imports, GPU 0 Isolation, and Model Loading Complete!")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 479/479 [00:03<00:00, 140.03it/s]


Imports, GPU 0 Isolation, and Model Loading Complete!


In [2]:
import librosa
import soundfile as sf
import io
from datasets import load_dataset, Dataset, Audio
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 2. LOAD & PREPROCESS (MANUAL LOOP / TRUE BYPASS)
# ---------------------------------------------------------
print("Loading medical dataset...")
raw_dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", split="test")

# CRITICAL FIX: Instruct the dataset NOT to decode the audio automatically upon access.
raw_dataset = raw_dataset.cast_column("audio", Audio(decode=False))

processed_data = []

print("Processing audio manually (bypassing HF map)...")
# We use a standard Python loop wrapped in tqdm so you can actually SEE the progress
for item in tqdm(raw_dataset, desc="Extracting Features"):
    # Now, audio_dict just contains raw bytes or paths, safely bypassing torchcodec.
    audio_dict = item["audio"]
    
    # 1. Safely load the audio directly from raw bytes into memory
    if audio_dict.get("bytes") is not None:
        audio_array, sr = sf.read(io.BytesIO(audio_dict["bytes"]))
        # Force it to 16kHz for Whisper
        if sr != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
    else:
        # Fallback to loading from path
        audio_array, _ = librosa.load(audio_dict["path"], sr=16000)
    
    # Ensure audio_array is 1D (mono)
    if len(audio_array.shape) > 1:
        audio_array = librosa.to_mono(audio_array.T)
        
    # 2. Extract features and labels
    input_features = processor.feature_extractor(
        audio_array, sampling_rate=16000
    ).input_features[0]
    
    labels = processor.tokenizer(item["text"]).input_ids 
    
    # 3. Store the processed data
    processed_data.append({
        "input_features": input_features,
        "labels": labels
    })

print("Converting back to Hugging Face Dataset format...")
medical_dataset = Dataset.from_list(processed_data)

print("Dataset ready for training!")

Loading medical dataset...


Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Processing audio manually (bypassing HF map)...


Extracting Features: 100%|██████████| 3619/3619 [02:14<00:00, 26.81it/s]


Converting back to Hugging Face Dataset format...
Dataset ready for training!


In [3]:
# ---------------------------------------------------------
# 3. DATA COLLATOR
# ---------------------------------------------------------
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [4]:
# ---------------------------------------------------------
# 4. A100 OPTIMIZED TRAINING ARGUMENTS
# ---------------------------------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./medical_whisper_model",
    per_device_train_batch_size=8,  # Increased for A100
    gradient_accumulation_steps=4,  # Lowered since batch size is higher
    learning_rate=1e-4,             
    warmup_steps=50,
    max_steps=400,                  
    bf16=True,                      
    fp16=False,
    optim="adamw_8bit",
    eval_strategy="no",             
    save_steps=100,
    logging_steps=10,
    report_to=["none"],
    remove_unused_columns=False,
    label_names=["labels"],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=medical_dataset,
    data_collator=data_collator,
    processing_class=processor, # <--- FIXED: Replaced 'tokenizer' with 'processing_class'
)

In [ ]:
# ---------------------------------------------------------
# 5. START TRAINING
# ---------------------------------------------------------
print("Starting fine-tuning...")
trainer.train()

print("Saving final medical model...")
trainer.save_model("./medical_whisper_final")

Starting fine-tuning...


/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,9.253812
20,8.764970
30,7.559435
40,6.504767
50,5.691784
60,5.138905
70,4.867076
80,4.205500
90,3.971778
100,4.250310


/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during